# Sprint 1 — Data Exploration

**CadetX Heavy Supplier, Inventory & Warehouse Analytics**
Team: DA1 · DA2 · DS · Scrum Master: _[name]_

Purpose: load all 12 tables, profile them, verify the join model and quantify the first-pass
data quality picture. No cleaning or modelling here — Sprint 2 owns that.

Set `DATA_DIR` if the CSVs are not in `../../data`:

```bash
DATA_DIR=/path/to/csvs jupyter nbconvert --execute week-01/notebooks/data_exploration.ipynb
```

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

NOTEBOOK_DIR = Path.cwd()
DATA_DIR = Path(os.environ["DATA_DIR"]) if "DATA_DIR" in os.environ else NOTEBOOK_DIR.parent.parent / "data"
OUT_DIR = NOTEBOOK_DIR.parent / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = [
    "branches", "customers", "inventory_master", "invoices", "payments", "products",
    "purchase_orders_header", "purchase_orders_lines", "sales_orders_header",
    "sales_orders_lines", "stock_ledger", "suppliers",
]
print("data dir :", DATA_DIR)
print("output dir:", OUT_DIR)

data dir : /home/user/.doubao/agent_mode/workspace/.sessions/38453146730768898/attachments
output dir: /home/user/Doubao/chats/38453146730768898/sprint-01-pack/week-01/outputs


## 1. Load every table

All tables are small enough to hold in memory (largest is the 237k-row stock ledger).
Identifiers are read as text so leading zeros and ID shapes survive.

In [2]:
ID_COLS = {
    "branches": ["branch_id"], "customers": ["customer_id"], "inventory_master": ["product_id", "branch_id"],
    "invoices": ["invoice_id", "so_id", "customer_id", "branch_id"], "payments": ["payment_id", "invoice_id"],
    "products": ["product_id"], "purchase_orders_header": ["po_id", "supplier_id", "branch_id"],
    "purchase_orders_lines": ["po_id", "product_id"], "sales_orders_header": ["so_id", "customer_id", "branch_id"],
    "sales_orders_lines": ["so_id", "product_id"], "stock_ledger": ["movement_id", "product_id", "branch_id"],
    "suppliers": ["supplier_id"],
}

frames = {}
for name in FILES:
    df = pd.read_csv(DATA_DIR / f"{name}.csv", dtype={c: "string" for c in ID_COLS[name]})
    frames[name] = df
    print(f"{name:26s} {df.shape[0]:>7,} rows  {df.shape[1]:>2} cols")

DATE_COLS = {
    "sales_orders_header": ["order_date", "delivery_date"],
    "purchase_orders_header": ["order_date", "expected_delivery_date", "received_date"],
    "invoices": ["invoice_date", "due_date"],
    "payments": ["payment_date"],
    "stock_ledger": ["movement_date"],
    "customers": ["customer_since", "last_purchase_date"],
    "products": ["last_purchase_date"],
}
for name, cols in DATE_COLS.items():
    for c in cols:
        frames[name][c] = pd.to_datetime(frames[name][c], errors="coerce")

branches                         6 rows  13 cols
customers                      500 rows  15 cols
inventory_master               180 rows   8 cols
invoices                    18,033 rows  10 cols
payments                    19,257 rows   5 cols
products                        30 rows  23 cols
purchase_orders_header      24,000 rows  10 cols


purchase_orders_lines      155,495 rows   9 cols
sales_orders_header         20,000 rows  11 cols
sales_orders_lines         130,402 rows   9 cols


stock_ledger               237,230 rows   9 cols
suppliers                        8 rows  12 cols


## 2. Table profile — shape, nulls, cardinality

`table_summary.csv` is committed as the sprint's structural evidence.

In [3]:
rows = []
for name, df in frames.items():
    nulls = int(df.isna().sum().sum())
    rows.append({
        "table": name,
        "rows": len(df),
        "columns": df.shape[1],
        "null_cells": nulls,
        "null_rate_%": round(nulls / (df.shape[0] * df.shape[1]) * 100, 3),
        "duplicate_rows": int(df.duplicated().sum()),
    })
summary = pd.DataFrame(rows)
summary.to_csv(OUT_DIR / "table_summary.csv", index=False)
summary

,table,rows,columns,null_cells,null_rate_%,duplicate_rows
0,branches,6,13,0,0.000,0
1,customers,500,15,0,0.000,0
2,inventory_master,180,8,0,0.000,0
3,invoices,18033,10,0,0.000,0
4,payments,19257,5,0,0.000,0
5,products,30,23,0,0.000,0
6,purchase_orders_header,24000,10,2370,0.988,0
7,purchase_orders_lines,155495,9,0,0.000,0
8,sales_orders_header,20000,11,0,0.000,0
9,sales_orders_lines,130402,9,0,0.000,0


## 3. Keys, grain and time coverage

Before joining anything, confirm the key resolution and the date window each table can support.

In [4]:
KEY_CHECKS = {
    "branches": ["branch_id"],
    "customers": ["customer_id"],
    "inventory_master": ["product_id", "branch_id"],
    "products": ["product_id"],
    "suppliers": ["supplier_id"],
    "sales_orders_header": ["so_id"],
    "sales_orders_lines": ["so_id", "line_number"],
    "purchase_orders_header": ["po_id"],
    "purchase_orders_lines": ["po_id", "line_number"],
    "invoices": ["invoice_id"],          # expected to fail: known issue DQ-03
    "payments": ["payment_id"],
    "stock_ledger": ["movement_id"],
}

checks = []
for name, keys in KEY_CHECKS.items():
    df = frames[name]
    dup = int(df.duplicated(subset=keys).sum())
    date_col = next((c for c in ["order_date", "invoice_date", "payment_date", "movement_date"]
                     if c in df.columns), None)
    checks.append({
        "table": name, "key": " + ".join(keys), "rows": len(df),
        "duplicate_key_rows": dup,
        "date_from": df[date_col].min().date().isoformat() if date_col else "",
        "date_to": df[date_col].max().date().isoformat() if date_col else "",
    })

key_checks = pd.DataFrame(checks)
key_checks.to_csv(OUT_DIR / "key_date_checks.csv", index=False)
key_checks

,table,key,rows,duplicate_key_rows,date_from,date_to
0,branches,branch_id,6,0,,
1,customers,customer_id,500,0,,
2,inventory_master,product_id + branch_id,180,0,,
3,products,product_id,30,0,,
4,suppliers,supplier_id,8,0,,
5,sales_orders_header,so_id,20000,0,2019-01-01,2024-12-31
6,sales_orders_lines,so_id + line_number,130402,0,,
7,purchase_orders_header,po_id,24000,0,2019-01-01,2024-12-31
8,purchase_orders_lines,po_id + line_number,155495,0,,
9,invoices,invoice_id,18033,197,2019-01-03,2025-01-13


### Referential integrity

Checks that decide whether joins are safe: orphan lines, orphan payments, missing combinations.

In [5]:
so, sol = frames["sales_orders_header"], frames["sales_orders_lines"]
po, pol = frames["purchase_orders_header"], frames["purchase_orders_lines"]
inv, pay, led = frames["invoices"], frames["payments"], frames["stock_ledger"]
im, pr, br = frames["inventory_master"], frames["products"], frames["branches"]

print("orders without lines      :", (~so.so_id.isin(sol.so_id)).sum(), "/", len(so))
print("orphan sales lines        :", (~sol.so_id.isin(so.so_id)).sum(), "/", len(sol))
print("POs without lines         :", (~po.po_id.isin(pol.po_id)).sum(), "/", len(po))
print("orphan purchase lines     :", (~pol.po_id.isin(po.po_id)).sum(), "/", len(pol))
print("payments with no invoice  :", (~pay.invoice_id.isin(inv.invoice_id)).sum(), "/", len(pay))
print("invoice so_id resolves    :", inv.so_id.isin(so.so_id).all())
print("inventory combinations    :", len(im), "=", im.product_id.nunique(), "SKUs x", im.branch_id.nunique(), "branches")
print("products covered          :", im.product_id.nunique(), "/", len(pr))
print("negative ledger balances  :", (led.running_balance < 0).sum())

orders without lines      : 0 / 20000
orphan sales lines        : 0 / 130402
POs without lines         : 0 / 24000
orphan purchase lines     : 0 / 155495
payments with no invoice  : 0 / 19257
invoice so_id resolves    : True
inventory combinations    : 180 = 30 SKUs x 6 branches
products covered          : 30 / 30
negative ledger balances  : 0


### The invoice-ID defect, quantified

`invoice_id` looks like a primary key but is not. This check exists so nobody joins payments on it
alone.

In [6]:
dup_mask = inv.invoice_id.duplicated(keep=False)
print("duplicate invoice_id rows      :", int(inv.invoice_id.duplicated().sum()))
print("distinct invoice_id duplicated :", inv.loc[inv.invoice_id.duplicated(), "invoice_id"].nunique())

inv[dup_mask].sort_values("invoice_id").head(6)[
    ["invoice_id", "so_id", "customer_id", "invoice_date", "grand_total", "payment_status"]
]

duplicate invoice_id rows      : 197
distinct invoice_id duplicated : 196


,invoice_id,so_id,customer_id,invoice_date,grand_total,payment_status
14917,INV-105711,SO-901766,C0208,2021-11-20,1792683.6,Paid
8324,INV-105711,SO-549003,C0191,2024-12-15,676549.2,Paid
14643,INV-106596,SO-807417,C0281,2024-01-25,47318.0,Paid
9894,INV-106596,SO-399413,C0409,2021-06-16,1540731.4,Paid
1712,INV-111054,SO-233671,C0072,2023-04-11,1527730.4,Paid
1706,INV-111054,SO-316263,C0313,2019-08-02,365.8,Paid


## 4. Stock level vs configured thresholds

The central finding of the sprint: the quantity in stock and the parameters that police it are
not on the same scale. `inventory_flags.csv` records the result per SKU–branch, including whether
the ledger reconciles to `current_stock`.

In [7]:
im2 = im.merge(pr[["product_id", "product_name", "category", "unit_cost", "max_stock_level"]], on="product_id")
im2["stock_value"] = im2.current_stock * im2.unit_cost
im2["vs_max"] = im2.current_stock / im2.max_stock
im2["vs_safety"] = im2.current_stock / im2.safety_stock
im2["vs_opening"] = im2.current_stock / im2.opening_stock

print("current_stock  mean / median:", round(im2.current_stock.mean()), "/", round(im2.current_stock.median()))
print("max_stock      mean / median:", round(im2.max_stock.mean()), "/", round(im2.max_stock.median()))
print("opening_stock  mean / median:", round(im2.opening_stock.mean()), "/", round(im2.opening_stock.median()))
print("median current/max ratio     :", round(im2.vs_max.median(), 1))
print("combinations above max_stock :", int((im2.current_stock > im2.max_stock).sum()), "/", len(im2))
print("combinations below safety    :", int((im2.current_stock < im2.safety_stock).sum()), "/", len(im2))
print("inventory value              :", f"{im2.stock_value.sum():,.0f}")

current_stock  mean / median: 107240 / 107790
max_stock      mean / median: 322 / 310
opening_stock  mean / median: 183 / 178
median current/max ratio     : 349.4
combinations above max_stock : 180 / 180
combinations below safety    : 0 / 180
inventory value              : 281,292,677,410


Reconcile the ledger to the snapshot. If closing balance equals `current_stock` for every
combination, the snapshot is internally consistent and the problem is the threshold values.

In [8]:
last_balance = (led.sort_values("movement_date")
                  .groupby(["product_id", "branch_id"]).running_balance.last()
                  .rename("ledger_closing"))
recon = im2.set_index(["product_id", "branch_id"]).join(last_balance)
recon["recon_diff"] = recon.current_stock - recon.ledger_closing
print("combinations checked        :", len(recon))
print("max absolute difference     :", int(recon.recon_diff.abs().max()))
print("ratio current/ledger == 1.0 :", bool((recon.current_stock / recon.ledger_closing == 1).all()))

combinations checked        : 180
max absolute difference     : 0
ratio current/ledger == 1.0 : True


In [9]:
flags = recon.reset_index()[[
    "product_id", "product_name", "category", "branch_id", "current_stock", "unit_cost", "stock_value",
    "opening_stock", "reorder_level", "safety_stock", "max_stock", "max_stock_level", "vs_max", "vs_safety",
    "ledger_closing", "recon_diff",
]].copy()
flags["flag_configured"] = np.where(flags.current_stock > flags.max_stock, "overstock",
                            np.where(flags.current_stock < flags.safety_stock, "below safety", "normal"))
flags.to_csv(OUT_DIR / "inventory_flags.csv", index=False)
flags.head()

,product_id,product_name,category,branch_id,current_stock,unit_cost,stock_value,opening_stock,reorder_level,safety_stock,max_stock,max_stock_level,vs_max,vs_safety,ledger_closing,recon_diff,flag_configured
0,P001,Hydraulic Pump HP-300,Hydraulic,DEL001,115928,18950,2196835600,196,59,46,340,40,340.964706,2520.173913,115928,0,overstock
1,P001,Hydraulic Pump HP-300,Hydraulic,PUN001,109389,18950,2072921550,198,70,43,349,40,313.435530,2543.930233,109389,0,overstock
2,P001,Hydraulic Pump HP-300,Hydraulic,CHN001,99373,18950,1883118350,187,62,44,311,40,319.527331,2258.477273,99373,0,overstock
3,P001,Hydraulic Pump HP-300,Hydraulic,HYD001,108772,18950,2061229400,226,71,65,417,40,260.844125,1673.415385,108772,0,overstock
4,P001,Hydraulic Pump HP-300,Hydraulic,KOL001,103808,18950,1967161600,263,94,58,475,40,218.543158,1789.793103,103808,0,overstock


## 5. Purchases vs delivered sales

If purchases run consistently ahead of sales, stock builds up by construction. Compare totals,
then check whether the pattern is SKU-specific or systemic.

In [10]:
delivered = so[so.order_status == "Delivered"]
valid_so = set(delivered.so_id)
received_po = set(po[po.po_status == "Received"].po_id)

sold = sol[sol.so_id.isin(valid_so)].groupby("product_id").quantity.sum().rename("units_sold")
bought = pol[pol.po_id.isin(received_po)].groupby("product_id").quantity.sum().rename("units_bought")

ps = pd.concat([sold, bought], axis=1).fillna(0)
ps["bought_to_sold"] = ps.units_bought / ps.units_sold.replace(0, np.nan)
ps = ps.join(pr.set_index("product_id")[["product_name", "category"]])
ps = ps.sort_values("bought_to_sold", ascending=False).round(2)

print("units received:", f"{ps.units_bought.sum():,.0f}")
print("units sold    :", f"{ps.units_sold.sum():,.0f}")
print("ratio         :", round(ps.units_bought.sum() / ps.units_sold.sum(), 2))
print("SKU ratio range:", round(ps.bought_to_sold.min(), 1), "-", round(ps.bought_to_sold.max(), 1))

ps.reset_index().to_csv(OUT_DIR / "purchase_vs_sales.csv", index=False)
ps.head(10)

units received: 22,387,123
units sold    : 1,234,939
ratio         : 18.13
SKU ratio range: 17.4 - 19.2


,units_sold,units_bought,bought_to_sold,product_name,category
product_id,,,,,
P024,39486,757049,19.17,Track Roller TR-60,Undercarriage
P014,40574,764100,18.83,Transmission Assembly TA-800,Transmission
P023,40276,758185,18.82,Idler Wheel IW-55,Undercarriage
P005,40480,759066,18.75,Fuel Injector FI-220,Engine
P002,40917,764701,18.69,Engine Oil Filter OF-90,Filters
P017,40920,761754,18.62,Fan Belt FB-30,Belts
P013,39927,740939,18.56,Seal Kit SK-15,Seals
P025,41091,757080,18.42,Cabin Glass CG-20,Cabin
P018,40752,748671,18.37,Wheel Rim WR-24,Undercarriage


## 6. Supplier promised vs actual delivery

`reliability_score` is a supplied opinion; the on-time rate below is measured. Compare them.

In [11]:
po2 = po[po.po_status == "Received"].copy()
po2["actual_lt"] = (po2.received_date - po2.order_date).dt.days
po2["promised_lt"] = po2.supplier_id.map(frames["suppliers"].set_index("supplier_id").lead_time_days)
po2["on_time"] = po2.actual_lt <= po2.promised_lt

sup = (po2.groupby("supplier_id")
       .agg(orders=("po_id", "count"), avg_actual_lt=("actual_lt", "mean"),
            avg_promised_lt=("promised_lt", "mean"), on_time_rate=("on_time", "mean"))
       .join(frames["suppliers"].set_index("supplier_id")[["supplier_name", "reliability_score"]])
       .round(3))
sup["on_time_rate"] = sup.on_time_rate.round(3)
sup = sup.sort_values("on_time_rate", ascending=False)

print("overall on-time rate:", round(po2.on_time.mean() * 100, 1), "%")
print("correlation(reliability_score, on_time_rate):",
      round(sup.reliability_score.corr(sup.on_time_rate), 3))
sup.reset_index().to_csv(OUT_DIR / "supplier_ontime.csv", index=False)
sup

overall on-time rate: 84.9 %
correlation(reliability_score, on_time_rate): -0.318


,orders,avg_actual_lt,avg_promised_lt,on_time_rate,supplier_name,reliability_score
supplier_id,,,,,,
SUP0005,2754,28.485,28.0,0.853,Tianjin OEM Supplies,3
SUP0006,2752,10.501,10.0,0.852,Qingdao Distributor Supplies,3
SUP0002,2635,10.516,10.0,0.850,Guangzhou Local Vendor Supplies,5
SUP0003,2723,16.531,16.0,0.850,Shanghai Distributor Supplies,4
SUP0001,2653,19.496,19.0,0.849,Shenzhen OEM Supplies,5
SUP0004,2664,18.497,18.0,0.848,Ningbo Distributor Supplies,3
SUP0007,2727,12.528,12.0,0.847,Suzhou Local Vendor Supplies,5
SUP0008,2722,26.546,26.0,0.842,Hangzhou OEM Supplies,4


## 7. Revenue concentration (ABC seed)

A quick Pareto check hands Sprint 4 a defensible prioritisation order.

In [12]:
rev = sol[sol.so_id.isin(valid_so)].groupby("product_id").line_total.sum().sort_values(ascending=False)
share = rev / rev.sum()
cum = share.cumsum()
abc = pd.DataFrame({"revenue": rev, "share": share.round(4), "cum_share": cum.round(4)})
abc["class"] = np.where(cum <= 0.8, "A", np.where(cum <= 0.95, "B", "C"))
print(abc["class"].value_counts().to_string())
print("A-class revenue share:", round(abc.loc[abc["class"] == "A", "share"].sum() * 100, 1), "%")

class
C    14
A     9
B     7
A-class revenue share: 78.5 %


## 8. Sprint 1 summary (printed for the review)

Numbers below are computed at run time, so this cell never goes stale.

In [13]:
cogs = (sol[sol.so_id.isin(valid_so)].merge(pr[["product_id", "unit_cost"]], on="product_id")
        .eval("quantity * unit_cost").sum())
rev_total = delivered.total_order_value.sum()
stock_value = im2.stock_value.sum()
monthly_branch_sales = (sol[sol.so_id.isin(valid_so)]
                        .merge(so[["so_id", "branch_id"]], on="so_id")
                        .groupby(["product_id", "branch_id"]).quantity.sum() / 72)
cover_days = im.current_stock.sum() / monthly_branch_sales.sum() * 30.44

print("=" * 62)
print("SPRINT 1 SUMMARY")
print("=" * 62)
print(f"rows analysed              : {sum(len(df) for df in frames.values()):,}")
print(f"revenue (delivered)        : {rev_total:,.0f}")
print(f"COGS                       : {cogs:,.0f}")
print(f"gross margin               : {(1 - cogs / rev_total) * 100:.1f}%")
print(f"inventory value            : {stock_value:,.0f}")
print(f"purchase / sales multiple  : {ps.units_bought.sum() / ps.units_sold.sum():.0f}x")
print(f"days of cover (branch view): {cover_days:,.0f}")
print(f"supplier on-time rate      : {po2.on_time.mean() * 100:.1f}%")
print(f"overstock combos (config)  : {(im.current_stock > im.max_stock).sum()} / {len(im)}")
print("=" * 62)

SPRINT 1 SUMMARY
rows analysed              : 605,141
revenue (delivered)        : 22,958,987,430
COGS                       : 18,008,980,690
gross margin               : 21.6%
inventory value            : 281,292,677,410
purchase / sales multiple  : 18x
days of cover (branch view): 34,258
supplier on-time rate      : 84.9%
overstock combos (config)  : 180 / 180
